In [ ]:
import pandas as pd
import anndata as ad
import scirpy as ir
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import os
import torch


#import tcrdist
import wandb

#import tensorflow as tf
import numpy as np

from torch import nn

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

from sklearn.preprocessing import LabelEncoder
from Levenshtein import distance as edit_distance
from collections import defaultdict

#import warnings
import random
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import os

import Levenshtein

## Setup

### Read in files

In [ ]:
wandb_usage = False

In [ ]:
if wandb_usage:
    WANDB_API_CHANGE_AS_NEEDED = "YOUR_WANDB_KEY"
    wandb.login(key=WANDB_API_CHANGE_AS_NEEDED)
else:
    print("Wandb unused.")

In [ ]:
np.random.seed(42)

In [ ]:
MODEL_DIR = "/PATH/TO/checkpointed_models_dir"
MODEL_SUBDIR = "/PATH/TO/final_version_dir"

DATA_DIR = "/PATH/TO/datadir"
ANNDATA_DIR = "/PATH/TO/anndatadir"

os.makedirs(os.path.join(MODEL_DIR, MODEL_SUBDIR), exist_ok=True)

In [ ]:
# split adata.obs into train and test in another file, read them in now (and don't use test until later)
df_train = pd.read_csv(DATA_DIR+"/trainset.csv")
df_test = pd.read_csv(DATA_DIR+"/testset.csv")
df_val = pd.read_csv(DATA_DIR+"/valset.csv")

df_train.head()

In [ ]:
# Use the same AnnData object here that you use in split_train_test.ipynb
adata = sc.read(ANNDATA_DIR+"/your_tcr_only_object.h5ad")
adata.obs.head()

### Define hyperparameters and vocab

In [ ]:
# Hyperparameters

LR = 1e-3
BATCH_SIZE = 256
GENE_EMB_DIM = 64
AA_EMB_DIM = 128
NUM_EPOCHS = 50
MAX_LEN_CHAIN = 32
MAX_NUM_CHAINS = 1
CNN_DIM = 32
KERNEL_SIZE = 3
OUT_DIM = 256

SEQ_WEIGHT_LOSS = 1.0
GENE_WEIGHT_LOSS = 0.15

BUILD_EDIT_T = 0.2
BUILD_EDIT_DIST = 2
MAX_NEIGHBORS = 25

In [ ]:
# define the recognized amino acids
AA_VOCAB = [
    'A','C','D','E','F','G','H','I','K','L','M','N',
    'P','Q','R','S','T','V','W','Y'
]

PAD_IDX = 0
MASK_IDX = 1 # for masking chains
JUNK_IDX = 2 # for any AAs not in the vocab that may appear

AA_TO_IDX = {aa:i+2+1 for i, aa in enumerate(AA_VOCAB)}  # 1-based

AA_TO_IDX["<PAD>"] = PAD_IDX
AA_TO_IDX["<MASK>"] = MASK_IDX
AA_TO_IDX["<JUNK>"] = JUNK_IDX
VOCAB_SIZE = max(AA_TO_IDX.values()) + 1

In [ ]:
def pad_chains(chains, max_len):
    """ Helper to pad chains to a specified length """
    return [ chain + [PAD_IDX]*(max_len - len(chain)) for chain in chains ]

def tokenize_aa_seperate(seq, max_len, max_chains):
    """
    Tokenizes each chain of the CDR3 amino acid sequence

    Arguments:
        seq: the CDR3 sequence (CDR3 TRA, CDR3 TRB, etc.)
        max_len: maximum length of chains (chains shorter than this are padded to max_len)
        max_chains: maximum number of chains allowed (to account for dual-chain TRAs or TRBs)
    """
    chains = []

    fill_missing_chains = PAD_IDX # NULL_IDX isnt ignored later so the model would "learn to generate nullity"
                                    # PAD_IDX is ignored so model will "ignore nullity"

    if seq is None or pd.isna(seq) or seq.upper() in ["<NA>", ""]:
        chains = [[fill_missing_chains]*max_len] * max_chains
        return chains

    for s in seq.split(";"):
        s = s.strip()
        if not s:
            continue
        chain = [AA_TO_IDX.get(aa.upper(), JUNK_IDX) for aa in s]
        if len(chain) < max_len:
            chain += [PAD_IDX] * (max_len - len(chain))
        else:
            chain = chain[:max_len]
        chains.append(chain)

    # pad missing chains to reach max_chains
    while len(chains) < max_chains:
        chains.append([fill_missing_chains]*max_len)

    return chains

In [ ]:
def depaired_array(df, col, seperator=';'):
    """
    Helper to account for possible dual chain format seperated by a seperator symbol

    Arguments:
        df: dataframe containing the (possibly) dual chain TRA or TRB
        col: column in df containing the chain information
    Returns:
        sorted list of all unique individual values 
        (["apple", "banana;orange", "orange;pear"] -> ["apple", "banana", "orange", pear])
    """
    de_paireds = []
    for x in df[col].dropna().unique():  # drop NaN first
        if ';' in x:
            de_paireds.extend(x.split(seperator))
        else:
            de_paireds.append(x)
    return sorted(list(set(de_paireds)))

In [ ]:
VGENE_TRA_VOCAB = depaired_array(adata.obs, 'v_gene_t_tra')
JGENE_TRA_VOCAB = depaired_array(adata.obs, 'j_gene_t_tra')
VGENE_TRB_VOCAB = depaired_array(adata.obs, 'v_gene_t_trb')
JGENE_TRB_VOCAB = depaired_array(adata.obs, 'j_gene_t_trb')
print(VGENE_TRA_VOCAB)
print(JGENE_TRA_VOCAB)
print(VGENE_TRB_VOCAB)
print(JGENE_TRB_VOCAB)

#NULL_GENE_IDX = 0
#PAD_GENE_IDX = 0

VGENE_TRA_TO_IDX = {aa:i+0 for i, aa in enumerate(VGENE_TRA_VOCAB)} 
VGENE_TRB_TO_IDX = {aa:i+0 for i, aa in enumerate(VGENE_TRB_VOCAB)}

JGENE_TRA_TO_IDX = {aa:i+0 for i, aa in enumerate(JGENE_TRA_VOCAB)}
JGENE_TRB_TO_IDX = {aa:i+0 for i, aa in enumerate(JGENE_TRB_VOCAB)}

In [ ]:
VGENE_TRA_TO_IDX.get('TRAV10')

In [ ]:
for gene in "tratrbtrc".split(","):
    print(gene)

## Dataset

### Definition and initialization of cell dataset

In [ ]:
class CellDataset(Dataset):
    def __init__(self, adata_obs, max_len, max_chains,
                 edit_neighbors,
                 cell_to_seq,
                 seq_to_cells,
                 to_pad=True):
        self.adata_obs = adata_obs
        self.max_len = max_len
        self.max_chains=max_chains
        self.to_pad = to_pad
        self.edit_neighbors = edit_neighbors
        self.cell_to_seq = cell_to_seq
        self.seq_to_cells = seq_to_cells
    
    def pad(self, tokens):
        if not self.to_pad:
            return tokens
        if len(tokens) >= self.max_len:
            return tokens[:self.max_len]
        else:
            return tokens + [PAD_IDX]*(self.max_len - len(tokens))
        
    def __getitem__(self, idx):
        row = self.adata_obs.iloc[idx]

        if self.to_pad:
            tra_tokens = tokenize_aa_seperate(row.cdr3_t_tra, max_len=self.max_len, max_chains=self.max_chains)
            trb_tokens = tokenize_aa_seperate(row.cdr3_t_trb, max_len=self.max_len, max_chains=self.max_chains)

        else:
            tra_tokens = tokenize_aa_seperate(row.cdr3_t_tra, to_pad=False)
            trb_tokens = tokenize_aa_seperate(row.cdr3_t_trb, to_pad=False)

        def multi_hot_genes(gene_str, gene_to_idx):
            zero_init_multihot = torch.zeros(max(gene_to_idx.values()) + 1)
            if pd.isna(gene_str) or gene_str.strip() == "" or gene_str.strip() =="<NA>":
                return zero_init_multihot
            gene_indices = [gene_to_idx[gene] for gene in gene_str.split(';')]
            zero_init_multihot[gene_indices] = 1
            return zero_init_multihot

        # Map V/J genes to indices safely
        tra_v_multihot = multi_hot_genes(row.v_gene_t_tra, VGENE_TRA_TO_IDX)
        trb_v_multihot = multi_hot_genes(row.v_gene_t_trb, VGENE_TRB_TO_IDX)
        tra_j_multihot = multi_hot_genes(row.j_gene_t_tra, JGENE_TRA_TO_IDX)
        trb_j_multihot = multi_hot_genes(row.j_gene_t_trb, JGENE_TRB_TO_IDX)

        return {
            "idx": idx,
            "sample_id": row['sample_id'],
            "cell_idx": row['cell_idx'],
            "tra_tokens": torch.tensor(tra_tokens, dtype=torch.long),
            "trb_tokens": torch.tensor(trb_tokens, dtype=torch.long),
            
            "tra_v_multihot": tra_v_multihot.float(),
            "trb_v_multihot": trb_v_multihot.float(),
            "tra_j_multihot": tra_j_multihot.float(),
            "trb_j_multihot": trb_j_multihot.float()
        }

    def __len__(self):
        return len(self.adata_obs)

def collate_multichain(batch, max_len=32):
    B = len(batch)
    
    # Allocate tensors
    tra_tensor = torch.full((B, MAX_NUM_CHAINS, max_len), fill_value=PAD_IDX, dtype=torch.long)
    trb_tensor = torch.full((B, MAX_NUM_CHAINS, max_len), fill_value=PAD_IDX, dtype=torch.long)

    for i, item in enumerate(batch):
        for j, chain in enumerate(item["tra_tokens"]):
            tra_tensor[i, j, :len(chain)] = torch.tensor(chain, dtype=torch.long)
        for j, chain in enumerate(item["trb_tokens"]):
            trb_tensor[i, j, :len(chain)] = torch.tensor(chain, dtype=torch.long)

    # Stack everything else normally
    out = {
        "idx": [x["idx"] for x in batch],
        "sample_id": [x["sample_id"] for x in batch],
        "cell_idx": [x["cell_idx"] for x in batch],
        "tra_tokens": tra_tensor,
        "trb_tokens": trb_tensor,
        "tra_v_multihot": torch.stack([x["tra_v_multihot"] for x in batch]),
        "trb_v_multihot": torch.stack([x["trb_v_multihot"] for x in batch]),
        "tra_j_multihot": torch.stack([x["tra_j_multihot"] for x in batch]),
        "trb_j_multihot": torch.stack([x["trb_j_multihot"] for x in batch]),
    }
    return out

In [ ]:
def worker_init_fn(_):
    import warnings
    warnings.filterwarnings(
        "ignore",
        message="To copy construct from a tensor"
    )

In [ ]:
# Drop NAs upstream
df_train_valid = df_train[df_train["cdr3_t_trb"].notna() & (df_train["cdr3_t_trb"].str.len() > 0)].reset_index(drop=True)
df_val_valid   = df_val[df_val["cdr3_t_trb"].notna() & (df_val["cdr3_t_trb"].str.len() > 0)].reset_index(drop=True)
df_test_valid   = df_test[df_test["cdr3_t_trb"].notna() & (df_test["cdr3_t_trb"].str.len() > 0)].reset_index(drop=True)

# Extract sequences
trb_seqs_train = df_train_valid["cdr3_t_trb"].tolist()
trb_seqs_val   = df_val_valid["cdr3_t_trb"].tolist()
trb_seqs_test = df_test_valid["cdr3_t_trb"].tolist()

# Collapse duplicates and map back
trb_seqs_train_unique, inv_train = np.unique(trb_seqs_train, return_inverse=True)
trb_seqs_val_unique, inv_val     = np.unique(trb_seqs_val, return_inverse=True)
trb_seqs_test_unique, inv_test = np.unique(trb_seqs_test, return_inverse=True)

# Build mapping: cell -> sequence, sequence -> cells
cell_to_seq_train = inv_train  # positional indices
seq_to_cells_train = defaultdict(list)
for pos, seq_idx in enumerate(inv_train):
    seq_to_cells_train[seq_idx].append(pos)

cell_to_seq_val = inv_val
seq_to_cells_val = defaultdict(list)
for pos, seq_idx in enumerate(inv_val):
    seq_to_cells_val[seq_idx].append(pos)

cell_to_seq_test = inv_test
seq_to_cells_test = defaultdict(list)
for pos, seq_idx in enumerate(inv_test):
    seq_to_cells_test[seq_idx].append(pos)

### Create edit neighbors for contrastive

In [ ]:
def build_edit_neighbors(seqs, raw_edit_dist_mode=False, dist=1, max_neighbors=50):
    """
    Samples edit distance neighbors for the contrastive model to train off of

    Args:
        seqs: list of AA strings (len N)
        raw_edit_dist_mode: whether to use raw edit distance or a proportional metric for a similarity ceiling
        dist: the distance to set the ceiling at, above which no cell can be a neighbor
            if raw_edit_dist_mode == False: ceiling becomes (edit distance of a pair's CDR3s)/(max length among the pair) <= dist
            if raw_edit_dist_mode == True: ceiling becomes (edit distance of a pair's CDR3s) <= dist
        max_neighbors: the maximum number of neighbors a single cell can have
    Returns: 
        neighbors -> list of neighbors to a single T cell (if any)
    """
    neighbors = defaultdict(list)
    #negatives = []
    N = len(seqs)

    for i in range(N):
        if len(seqs[i]) <= 0:
            continue
        for j in range(i + 1, N):
            maxlen = max(len(seqs[i]), len(seqs[j]))
            if maxlen <= 0:
                continue
            
            if raw_edit_dist_mode:
                if edit_distance(seqs[i], seqs[j]) <= dist:
                    neighbors[i].append(j)
                    neighbors[j].append(i)

            else:
                if abs(len(seqs[i]) - len(seqs[j])) / maxlen > dist:
                    continue
                if edit_distance(seqs[i], seqs[j]) / maxlen  <= dist:
                    neighbors[i].append(j)
                    neighbors[j].append(i)
                    
        """
        # random neighbors, should perform badly
        # rand_arr = [random.randint(0, N-1) for _ in range(random.randint(4, 7))]
        # assign random (albation for random neighbors)
        # neighbors[i] = rand_arr
        
        # far edit distance neighbors, should peform even worse
        rand_arr = []
        if len(negatives) > 0:
            rand_arr = random.sample(negatives, k=random.randint(1, min(7, len(negatives)-1)))
        neighbors[i] = rand_arr
        for j in rand_arr:
            if i not in neighbors[j]:
                neighbors[j].append(i)
        """
        
        neighbors[i] = random.sample(neighbors[i], min(len(neighbors[i]), max_neighbors))
        
    return neighbors

In [ ]:
# Build neighbors on unique sequences
edit_neighbors_train = build_edit_neighbors(trb_seqs_train_unique, raw_edit_dist_mode=False, dist=0.225, max_neighbors=MAX_NEIGHBORS)
edit_neighbors_val   = build_edit_neighbors(trb_seqs_val_unique, raw_edit_dist_mode=False, dist=0.225, max_neighbors=MAX_NEIGHBORS)
edit_neighbors_test = build_edit_neighbors(trb_seqs_test_unique, raw_edit_dist_mode=False, dist=0.225, max_neighbors=MAX_NEIGHBORS)

### Dataset (cont.)

In [ ]:
# Create datasets
train_dataset = CellDataset(
    df_train_valid,
    max_len=MAX_LEN_CHAIN,
    max_chains=MAX_NUM_CHAINS,
    edit_neighbors=edit_neighbors_train,
    cell_to_seq=cell_to_seq_train,
    seq_to_cells=seq_to_cells_train
)

val_dataset = CellDataset(
    df_val_valid,
    max_len=MAX_LEN_CHAIN,
    max_chains=MAX_NUM_CHAINS,
    edit_neighbors=edit_neighbors_val,
    cell_to_seq=cell_to_seq_val,
    seq_to_cells=seq_to_cells_val
)

test_dataset = CellDataset(
    df_test_valid,
    max_len=MAX_LEN_CHAIN,
    max_chains=MAX_NUM_CHAINS,
    edit_neighbors=edit_neighbors_test, # unused in masked reconstruction evaluation
    cell_to_seq=cell_to_seq_test,
    seq_to_cells=seq_to_cells_test
)

In [ ]:
# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_multichain,
    num_workers=4,
    worker_init_fn=worker_init_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_multichain,
    num_workers=4,
    worker_init_fn=worker_init_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_multichain,
    num_workers=4,
    worker_init_fn=worker_init_fn
)

In [ ]:
# Number of sequences with at least one neighbor
num_with_neighbors = sum(len(nbrs) > 0 for nbrs in edit_neighbors_test.values())
num_total_seqs = len(edit_neighbors_train)

print(f"Total unique sequences: {num_total_seqs}")
print(f"Sequences with neighbors: {num_with_neighbors}")
print(f"Fraction with neighbors: {num_with_neighbors / num_total_seqs:.3f}")

nbr_counts = [len(v) for v in edit_neighbors_train.values()]
print("Mean neighbors:", np.mean(nbr_counts))
print("Pct with zero neighbors:", np.mean(np.array(nbr_counts) == 0))
print("Median:", np.median(nbr_counts))
print("90th pct:", np.percentile(nbr_counts, 90))
print("99th pct:", np.percentile(nbr_counts, 99))

In [ ]:
print(str(len(train_dataset)) + " " + str(len(val_dataset)) + " " + str(len(test_dataset)))

## Contrastive model

### Model definition

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
device

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x, mask):
        # [B, L, 1]
        scores = self.score(x.permute(0, 2, 1))

        scores = scores.squeeze(-1)  # [B, L]
        scores = scores.masked_fill(~mask, -1e9)

        weights = torch.softmax(scores, dim=-1)  # [B, L]

        pooled = torch.sum(
            x * weights.unsqueeze(1), dim=-1
        )  # [B, D]

        return pooled

In [ ]:
class ContrastiveCellEncoder(nn.Module):
    def __init__(
        self,
        aa_vocab_size,
        aa_embed_dim=16,
        cnn_dim=64,
        kernel_size=5,
        out_dim=128,
    ):
        super().__init__()

        self.aa_embedding = nn.Embedding(
            aa_vocab_size, aa_embed_dim, padding_idx=PAD_IDX
        )

        self.cnn_tra = nn.Conv1d(
            aa_embed_dim, cnn_dim, kernel_size, padding=kernel_size // 2
        )
        self.cnn_trb = nn.Conv1d(
            aa_embed_dim, cnn_dim, kernel_size, padding=kernel_size // 2
        )

        self.proj = nn.Sequential(
            nn.Linear(2 * cnn_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )

        self.attn_tra = AttentionPool(cnn_dim)
        self.attn_trb = AttentionPool(cnn_dim)

    def encode_chain(self, tokens, cnn, attn_pool):
        assert tokens.dtype == torch.long
        assert tokens.min().item() >= 0, tokens.min()
        assert tokens.max().item() < self.aa_embedding.num_embeddings, (
            tokens.max().item(),
            self.aa_embedding.num_embeddings
        )

        # read in tokens
        B, C, L = tokens.shape
        tokens = tokens.view(B * C, L)

        mask = (tokens != PAD_IDX).unsqueeze(1)  # [B*C, 1, L]

        emb = self.aa_embedding(tokens)          # [B*C, L, D]
        emb = emb.permute(0, 2, 1)               # [B*C, D, L]

        conv = torch.relu(cnn(emb))     # [B*C, D, L]
        pos_mask = (tokens != PAD_IDX) # [B*C, L]

        pooled = attn_pool(conv, pos_mask)
        pooled = pooled.view(B, C, -1)

        chain_mask = (tokens.view(B, C, L) != PAD_IDX).any(dim=2)
        pooled = pooled * chain_mask.unsqueeze(-1)

        denom = chain_mask.sum(dim=1, keepdim=True).clamp(min=1)
        return pooled.sum(dim=1) / denom

    def forward(self, batch):
        alpha = self.encode_chain(
            batch["tra_tokens"], self.cnn_tra, self.attn_tra
        )
        beta = self.encode_chain(
            batch["trb_tokens"], self.cnn_trb, self.attn_trb
        )

        cell = torch.cat([alpha, beta], dim=-1)
        cell = self.proj(cell)
        cell = torch.nn.functional.normalize(cell, dim=-1)
        return cell  # [B, out_dim]

### Check rate of no-neighbors-found in contrastive train set

In [ ]:
import random
def sample_positive_batch(batch, dataset):
    pos_items = []
    num_fallback = 0
    N = len(dataset)

    for idx in batch["idx"]:
        idx = idx.item() if torch.is_tensor(idx) else idx

        seq_idx = dataset.cell_to_seq[idx]
        nbr_seqs = dataset.edit_neighbors.get(seq_idx, [])

        if len(nbr_seqs) > 0:
            nbr_seq = random.choice(nbr_seqs)
            pos_idx = random.choice(dataset.seq_to_cells[nbr_seq])
        else:
            candidates = list(range(N))
            candidates.remove(idx)
            pos_idx = random.choice(candidates)
            num_fallback += 1

        pos_items.append(dataset[pos_idx])

    return pos_items, num_fallback

batch = next(iter(train_loader))
_, fallback = sample_positive_batch(batch, train_dataset)
print(f"Fallback rate in batch: {fallback / len(batch['idx']):.3f}")

### Contrastive training objective important functions

In [ ]:
def aa_mask(tokens, p=0.125):
    """
    Mask out certain amino acids
    Args:
        tokens: [B, C, L]
        p: what percent of amino acids to mask
    Returns:
        out: [B, C, L], tokens object with applied masking
    
    """
    mask = (torch.rand_like(tokens.float()) < p) & (tokens != PAD_IDX)
    out = tokens.clone()
    out[mask] = MASK_IDX
    return out

# unused
def chain_dropout(tokens, p=0.5):
    """
    Args:
        tokens: [B, C, L]
    randomly zero out entire chains
    """
    B, C, L = tokens.shape
    drop = torch.rand(B, C, device=tokens.device) < p
    tokens = tokens.clone()
    tokens[drop] = PAD_IDX
    return tokens

In [ ]:
def contrastive_loss(z1, z2, temperature=0.1):
    """
    Args:
        z1, z2: [B, D], normalized
        temperature: temp. of the InfoNCE loss
    
    Implementation of the InfoNCE loss.
    """
    B = z1.size(0)

    z = torch.cat([z1, z2], dim=0)  # [2B, D]
    sim = torch.matmul(z, z.T)      # cosine since normalized
    sim /= temperature

    labels = torch.arange(B, device=z.device)
    labels = torch.cat([labels + B, labels], dim=0)

    mask = torch.eye(2 * B, device=z.device).bool()
    sim = sim.masked_fill(mask, -9e15)

    return torch.nn.functional.cross_entropy(sim, labels)

In [ ]:
device

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*To copy construct from a tensor.*",
)

### Model initialization

In [ ]:
best_loss = float("inf")
best_state = None

con_encoder = ContrastiveCellEncoder(
    len(AA_TO_IDX),
    aa_embed_dim=AA_EMB_DIM,
    cnn_dim=CNN_DIM,
    kernel_size=KERNEL_SIZE,
    out_dim=OUT_DIM,
).to(device)

optimizer = torch.optim.AdamW(
    con_encoder.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=3e-6
)

In [ ]:
total_params = sum(p.numel() for p in con_encoder.parameters())
trainable_params = sum(p.numel() for p in con_encoder.parameters() if p.requires_grad)
print(total_params)
print(trainable_params)

## Contrastive training loop

In [ ]:
run_name = "quick_test_thing"
if wandb_usage:
    wandb.init(project="TCR Embeddings", name=run_name)

In [ ]:
for epoch in range(NUM_EPOCHS):
    con_encoder.train()
    epoch_loss = 0.0
    epoch_fallback = 0
    epoch_count = 0 
    for batch in train_loader:
        
        
        batch = {k: v.to(device) if torch.is_tensor(v) else v
                 for k, v in batch.items()}

        optimizer.zero_grad()

        # anchor
        z1 = con_encoder(batch)

        # positive cells
        pos_items, num_fallback = sample_positive_batch(batch, train_dataset)
        pos_batch = collate_multichain(pos_items)
        pos_batch = {k: v.to(device) if torch.is_tensor(v) else v
                     for k, v in pos_batch.items()}

        # augment positives
        pos_batch["tra_tokens"] = aa_mask(
            pos_batch["tra_tokens"],
             0.2
        )
        pos_batch["trb_tokens"] = aa_mask(
            pos_batch["trb_tokens"],
             0.2
        )

        z2 = con_encoder(pos_batch)

        with torch.no_grad():
            pos_sim = torch.sum(z1 * z2, dim=1).mean()

            perm = torch.randperm(z1.size(0))
            neg_sim = torch.sum(z1 * z1[perm], dim=1).mean()

        loss = contrastive_loss(z1, z2)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    epoch_fallback += num_fallback
    epoch_count += len(batch["idx"])

    if wandb_usage:
        wandb.log({
            "fallback_rate": num_fallback / len(batch["idx"]),
            "train_loss": avg_loss,
            "lr": optimizer.param_groups[0]["lr"],
            "pos_cos_sim": pos_sim.item(),
            "neg_cos_sim": neg_sim.item(),
            "fallback_rate": epoch_fallback / epoch_count
        })

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_state = con_encoder.state_dict()
        torch.save(best_state, os.path.join(MODEL_DIR, MODEL_SUBDIR, run_name + '.pt'))
        print(f"Epoch {epoch+1}, loss={avg_loss:.4f} (new best)")
    else:
        print(f"Epoch {epoch+1}, loss={avg_loss:.4f}")

    scheduler.step()

if wandb_usage:
    wandb.finish()

## Masked reconstruction model definition 

In [ ]:
class TCRReconstructor(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        max_len=50
    ):
        super().__init__()
        
        # Embedding
        self.token_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, embed_dim))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=4*embed_dim,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Decoder head
        self.decoder = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, tokens, return_embeddings=False):
        """
        tokens: [B, L] (concatenated TRA + TRB)
        """
        B, L = tokens.shape
        
        # Embed
        x = self.token_embed(tokens)  # [B, L, embed_dim]
        x = x + self.pos_embed[:, :L, :]  # add positional encoding
        
        # Mask padding for transformer
        padding_mask = (tokens == PAD_IDX)
        
        # Transform
        h = self.transformer(x, src_key_padding_mask=padding_mask)  # [B, L, embed_dim]
        
        # Predict
        logits = self.decoder(h)  # [B, L, vocab_size]
        
        if return_embeddings:
            # Pool for evaluation (mean over non-padding positions)
            mask = ~padding_mask
            pooled = (h * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True)
            return logits, pooled
        
        return logits

In [ ]:
def aa_mask_with_positions(tokens, p=0.15):
    """
    Mask amino acids and return mask positions
    
    Args:
        tokens: [B, L] or [N, L]
        p: masking probability
    
    Returns:
        masked_tokens: tokens with MASK_IDX at masked positions
        mask_positions: boolean tensor [B, L] indicating which were masked
    """
    # Only mask actual amino acids (not PAD)
    maskable = (tokens != PAD_IDX) & (tokens >= 3)  # amino acids start at index 3
    
    # Sample positions to mask
    mask_positions = (torch.rand_like(tokens.float()) < p) & maskable
    
    # Apply masking
    masked_tokens = tokens.clone()
    masked_tokens[mask_positions] = MASK_IDX
    
    return masked_tokens, mask_positions

In [ ]:
def prepare_reconstruction_batch(batch):
    """
    Flatten batch to individual chains for reconstruction
    
    Args:
        batch:
            batch["tra_tokens"]: [B, C, L]
            batch["trb_tokens"]: [B, C, L]
    Returns:
        tokens: [N, L] where N = number of non-padding chains
    """
    
    all_chains = []
    
    B, C, L = batch["tra_tokens"].shape
    
    # Flatten TRA chains
    tra_flat = batch["tra_tokens"].view(B * C, L)
    for chain in tra_flat:
        if not (chain == PAD_IDX).all():  # skip fully padded
            all_chains.append(chain)
    
    # Flatten TRB chains
    trb_flat = batch["trb_tokens"].view(B * C, L)
    for chain in trb_flat:
        if not (chain == PAD_IDX).all():
            all_chains.append(chain)
    
    if len(all_chains) == 0:
        return None
    
    return torch.stack(all_chains)  # [N, L]

### Model initialization

In [ ]:
# Initialize
vocab_size = len(AA_TO_IDX)
mask_encoder = TCRReconstructor(
    vocab_size=vocab_size,
    embed_dim=AA_EMB_DIM,
    num_heads=4,
    num_layers=2,
    max_len=MAX_LEN_CHAIN  # adjust based on your max chain length
).to(device)

optimizer = torch.optim.AdamW(
    mask_encoder.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=3e-6
)

best_loss = float("inf")
best_state = None

## Masked reconstruction training loop

In [ ]:
run_name = "masked_run_correct_adata"
if wandb_usage:
    wandb.init(project="TCR Embeddings", name=run_name)

In [ ]:
# Training loop
for epoch in range(NUM_EPOCHS):
    mask_encoder.train()
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0
    num_batches = 0
    
    for batch_idx, batch in enumerate(train_loader):
        # Move to device
        batch = {k: v.to(device) if torch.is_tensor(v) else v
                 for k, v in batch.items()}
        
        # Flatten to individual chains
        tokens = prepare_reconstruction_batch(batch)
        
        if tokens is None or len(tokens) == 0:
            continue
        
        tokens = tokens.to(device)  # [N, L]
        
        # Mask positions
        masked_tokens, mask_positions = aa_mask_with_positions(tokens, p=0.15)
        
        # Forward pass
        logits = mask_encoder(masked_tokens)  # [N, L, vocab_size]
        
        # Compute loss (only on masked positions)
        loss = F.cross_entropy(
            logits.view(-1, vocab_size),
            tokens.view(-1),
            ignore_index=PAD_IDX,
            reduction='none'
        )
        
        # Apply mask to focus loss on masked positions only
        loss = loss.view_as(tokens)  # [N, L]
        masked_loss = (loss * mask_positions.float()).sum() / mask_positions.sum().clamp(min=1)
        
        # Compute accuracy on masked positions
        preds = logits.argmax(dim=-1)  # [N, L]
        correct = ((preds == tokens) & mask_positions).sum().item()
        total = mask_positions.sum().item()
        
        # Backprop
        optimizer.zero_grad()
        masked_loss.backward()
        torch.nn.utils.clip_grad_norm_(mask_encoder.parameters(), 1.0)
        optimizer.step()
        
        # Track metrics
        epoch_loss += masked_loss.item()
        epoch_correct += correct
        epoch_total += total
        num_batches += 1
    
    # Epoch summary
    avg_loss = epoch_loss / num_batches
    avg_acc = epoch_correct / epoch_total if epoch_total > 0 else 0
    
    scheduler.step()
    
    if wandb_usage:
        wandb.log({
            "epoch": epoch,
            "train_loss": avg_loss,
            "train_accuracy": avg_acc,
            "lr": optimizer.param_groups[0]["lr"],
        })
    
    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_state = mask_encoder.state_dict()
        torch.save(best_state, os.path.join(MODEL_DIR, MODEL_SUBDIR, run_name + '.pt'))
        print(f"Epoch {epoch+1}, loss={avg_loss:.4f}, acc={avg_acc:.3f} (new best)")
    else:
        print(f"Epoch {epoch+1}, loss={avg_loss:.4f}, acc={avg_acc:.3f}")

if wandb_usage:
    wandb.finish()

## Evaluation setup

In [ ]:
read_in_contrastive = True
read_in_reconstruct = True # optinally for if you have extant models and want to have them be read in to run this section only

### Contrastive model evaluation setup

In [ ]:
if read_in_contrastive:
    con_encoder = ContrastiveCellEncoder(
        len(AA_TO_IDX),
        aa_embed_dim=AA_EMB_DIM,
        cnn_dim=CNN_DIM,
        kernel_size=KERNEL_SIZE,
        out_dim=OUT_DIM,
    ).to(device)

    con_encoder.load_state_dict(torch.load('/PATH/TO/your_checkpoint.pt'))

### Add evaluation-relevant metadata and contrastive embeddings

In [ ]:
# Merge external metadata
metadata_df = df_test.copy()

all_embeddings = []
all_cell_idx = []
all_patients = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        emb = con_encoder(batch)  # [B, D]
        all_embeddings.append(emb.cpu())
        all_cell_idx.extend(batch['cell_idx'])
        all_patients.extend(batch['sample_id'])

# Create a DataFrame linking embeddings to cell_idx
con_emb_df = pd.DataFrame({'cell_idx': all_cell_idx, 'embedding_idx': np.arange(len(all_cell_idx))})
merged_df = con_emb_df.merge(metadata_df, on="cell_idx", how="left")
merged_df.head()

In [ ]:
emb = con_encoder(batch)
print(emb.norm(dim=1).mean(), emb.norm(dim=1).std())

In [ ]:
merged_df['cdr3_comb'] = (
    merged_df["cdr3_t_tra"].fillna("").astype(str)
    + "_"
    + merged_df["cdr3_t_trb"].fillna("").astype(str)
)
merged_df[['cdr3_t_tra', 'cdr3_t_trb', 'cdr3_comb', 'embedding_idx']].head()

In [ ]:
# condense labels

valid_labels = ["HV", "Non-SjD", "PrimarySjD", "SecondarySjD"]
assert((metadata_df["label"].isin(valid_labels)).all())

metadata_df['label_condensed'] = [
    'SjD' if (x == 'PrimarySjD' or x == 'SecondarySjD') else ('Non-SjD')
    for x in metadata_df['label']
]

merged_df['label_condensed'] = [
    'SjD' if (x == 'PrimarySjD' or x == 'SecondarySjD') else ('Non-SjD')
    for x in metadata_df['label']
]

In [ ]:
Z = torch.cat(all_embeddings, dim=0).numpy()
merged_df["emb"] = list(Z)
counts = merged_df["cdr3_comb"].value_counts()

In [ ]:
def compute_pair_features(df_eval, emb_key, n_pairs=2000, seed=None):
    """
    Samples pairs and compute various features + cosine similarity

    Args:
        df_eval: the dataframe to run evaluation on
        emb_key: key specifying what embedding to use
        n_pairs: how many pairs of TCR embeddings to sample
    Returns:
        pd.Dataframe(pairs_data): n_pairs of sampled TCR pairs, 
        as well as information about the each pair's shared or divergent fields (eg. gene usage, patient of origin)
    """
    pairs_data = []

    if seed is not None:
        np.random.seed(seed)
    
    seen_pairs = set()

    while len(pairs_data) < n_pairs:
        i, j = np.random.choice(len(df_eval), 2, replace=False)
        key = (min(i,j), max(i,j))
        if key in seen_pairs:
            continue
        seen_pairs.add(key)
        
        row_i = df_eval.iloc[i]
        row_j = df_eval.iloc[j]
        
        # Edit distance
        #edit_dist = Levenshtein.distance(row_i.cdr3_comb, row_j.cdr3_comb)
        edit_dist_tra= Levenshtein.distance(row_i.cdr3_t_tra, row_j.cdr3_t_tra)
        edit_dist_trb= Levenshtein.distance(row_i.cdr3_t_trb, row_j.cdr3_t_trb)
        edit_dist = edit_dist_tra + edit_dist_trb
        
        # Cosine similarity
        cos_sim = cosine_similarity([row_i[emb_key]], [row_j[emb_key]])[0, 0]
        
        # Length difference
        len_diff = abs(len(row_i.cdr3_comb) - len(row_j.cdr3_comb))
        
        # Same V gene? (TRB)
        same_v_trb = (row_i.v_gene_t_trb == row_j.v_gene_t_trb)
        
        # Same J gene? (TRB)
        same_j_trb = (row_i.j_gene_t_trb == row_j.j_gene_t_trb)

        # Same V gene? (paired)
        same_v_paired = (row_i.v_gene_t_trb == row_j.v_gene_t_trb) & (row_i.v_gene_t_tra == row_j.v_gene_t_tra)
        
        # Same J gene? (paired)
        same_j_paired = (row_i.j_gene_t_trb == row_j.j_gene_t_trb) & (row_i.j_gene_t_tra == row_j.j_gene_t_tra)
        
        # Same clonotype?
        same_clonotype = (row_i.comb_raw_clonotype_id_t == row_j.comb_raw_clonotype_id_t)
        
        # Same sample/patient?
        same_sample = (row_i.sample_id == row_j.sample_id)

        # Disease status
        disease_status = 'TBD'
        if row_i.label_condensed != row_j.label_condensed:
            disease_status = '1x (+), 1x (-)'
        elif row_i.label_condensed == 'Non-SjD':
            disease_status = '2x (-)'
        elif row_i.label_condensed == 'SjD':
            disease_status = '2x (+)'

        # Add all information
        pairs_data.append({
            'edit_dist': edit_dist,
            'edit_dist_tra': edit_dist_tra,
            'edit_dist_trb': edit_dist_trb,
            'cos_sim': cos_sim,
            'len_diff': len_diff,
            'same_v_trb': same_v_trb,
            'same_j_trb': same_j_trb,
            'same_v_paired': same_v_paired,
            'same_j_paired': same_j_paired,
            'same_clonotype': same_clonotype,
            'same_sample': same_sample,
            'disease_status': disease_status
        })
    
    return pd.DataFrame(pairs_data)

### Masked reconstruction evaluation setup

In [ ]:
# After training
if read_in_reconstruct:
    mask_encoder = TCRReconstructor(
        vocab_size=len(AA_TO_IDX),
        embed_dim=AA_EMB_DIM,
        num_heads=4,
        num_layers=2,
        max_len=MAX_LEN_CHAIN  # adjust based on your max chain length
    ).to(device)

    # After training reconstruction model
    mask_encoder.load_state_dict(torch.load('checkpointed_models/masked_learning/masked_run_correct_adata.pt'))

mask_encoder.eval()
all_embeddings = []
all_cell_idx = []

# load in masked recon. embeddings
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) if torch.is_tensor(v) else v 
                    for k, v in batch.items()}
        
        # Extract tokenized chains
        tra_tokens = batch['tra_tokens']  # [B, C, L]
        trb_tokens = batch['trb_tokens']  # [B, C, L]
        
        B, C, L = tra_tokens.shape
        
        # Process each cell
        batch_embeddings = []
        for b in range(B):
            chain_embeddings = []
            
            # Encode TRA chains
            for c in range(C):
                chain = tra_tokens[b, c]  # [L]
                if not (chain == PAD_IDX).all():
                    chain_input = chain.unsqueeze(0)  # [1, L]
                    _, emb = mask_encoder(chain_input, return_embeddings=True)
                    chain_embeddings.append(emb.squeeze(0))
            
            # Encode TRB chains
            for c in range(C):
                chain = trb_tokens[b, c]
                if not (chain == PAD_IDX).all():
                    chain_input = chain.unsqueeze(0)
                    _, emb = mask_encoder(chain_input, return_embeddings=True)
                    chain_embeddings.append(emb.squeeze(0))
            
            # Pool chains for this cell
            if len(chain_embeddings) > 0:
                cell_emb = torch.stack(chain_embeddings).mean(dim=0)
            else:
                cell_emb = torch.zeros(mask_encoder.token_embed.embedding_dim).to(device)
            
            batch_embeddings.append(cell_emb)
        
        all_embeddings.append(torch.stack(batch_embeddings).cpu())
        all_cell_idx.extend(batch['cell_idx'])


# Create mask embedding dataframe
mask_embeddings = torch.cat(all_embeddings, dim=0).numpy()
mask_emb_df = pd.DataFrame({
    'cell_idx': all_cell_idx,
    'emb_reconstruction': list(mask_embeddings)
})

### Generate complete pairwise evaluation data

In [ ]:
# Merge with existing df (which has contrastive embeddings)
merged_df = merged_df.merge(mask_emb_df, on='cell_idx', how='left')
merged_df['cdr3_comb'] = (
    merged_df["cdr3_t_tra"].fillna("").astype(str) + "_" +
    merged_df["cdr3_t_trb"].fillna("").astype(str)

)
# Now merged_df has BOTH:
# - merged_df['emb'] (contrastive)
# - merged_df['emb_reconstruction'] (reconstruction)

### Generate various clone size -based subsets

In [ ]:
# Run this to see the distribution
counts = merged_df["cdr3_comb"].value_counts()

for threshold in [1, 2, 3, 5, 10, 20]:
    good_clones = counts[counts >= threshold].index
    df_filtered = merged_df[merged_df["cdr3_comb"].isin(good_clones)]
    
    print(f"Threshold ≥ {threshold:2d}:")
    print(f"  Cells:      {len(df_filtered):4d} ({len(df_filtered)/len(merged_df)*100:5.1f}%)")
    print(f"  Clonotypes: {df_filtered['cdr3_comb'].nunique():4d}")
    print()

In [ ]:
# Filter for good clones (other thresholds included below)
counts = merged_df["cdr3_comb"].value_counts()
df_eval_above_5 = merged_df[merged_df["cdr3_comb"].isin(counts[counts >= 5].index)].copy()
print(f"Evaluation set: {len(df_eval_above_5)} cells, {df_eval_above_5['cdr3_comb'].nunique()} clonotypes")

df_eval_above_3 = merged_df[merged_df["cdr3_comb"].isin(counts[counts >= 3].index)].copy()
print(f"Evaluation set: {len(df_eval_above_3)} cells, {df_eval_above_3['cdr3_comb'].nunique()} clonotypes")

df_eval_all = merged_df.copy()
print(f"Evaluation set: {len(df_eval_all)} cells, {df_eval_all['cdr3_comb'].nunique()} clonotypes")

In [ ]:
df_to_pick = df_eval_all.copy()

valid_labels = ["HV", "Non-SjD", "PrimarySjD", "SecondarySjD"]
assert((df_to_pick["label"].isin(valid_labels)).all())

df_to_pick['label_condensed'] = [
    'SjD' if (x == 'PrimarySjD' or x == 'SecondarySjD') else ('Non-SjD')
    for x in df_to_pick['label']
]

## kNN clonotype purity

In [ ]:
# Assuming you have df_eval with both embeddings
# df_eval['emb_contrastive'] - from your contrastive model
# df_eval['emb_reconstruction'] - from reconstruction model

# k-NN purity for both
from sklearn.neighbors import NearestNeighbors

def compute_knn_purity(embeddings, labels, k=10):
    """
    For each point, find k nearest neighbors and check label purity
    """
    nbrs = NearestNeighbors(n_neighbors=k+1, metric='cosine').fit(embeddings)
    distances, indices = nbrs.kneighbors(embeddings)
    
    purities = []
    for i, neighbors in enumerate(indices):
        neighbors = neighbors[1:]  # exclude self
        same_label = (labels[neighbors] == labels[i]).sum()
        purities.append(same_label / k)

    return np.mean(purities)

In [ ]:
labels = df_to_pick['comb_raw_clonotype_id_t'].values

# Now compare
purity_contrastive = compute_knn_purity(
    np.stack(df_to_pick['emb'].values),
    df_to_pick['comb_raw_clonotype_id_t'].values,
    k=10
)

purity_reconstruction = compute_knn_purity(
    np.stack(df_to_pick['emb_reconstruction'].values),
    df_to_pick['comb_raw_clonotype_id_t'].values,
    k=10
)

print(f"Contrastive k-NN purity: {purity_contrastive:.3f}")
print(f"Reconstruction k-NN purity: {purity_reconstruction:.3f}")

## Pairwise correlation analysis

In [ ]:
df_to_pick.shape

In [ ]:
if len(df_to_pick) > 1000:
    df_pairs_contrastive = compute_pair_features(df_to_pick, 'emb', n_pairs=10000, seed=42)
    df_pairs_recon = compute_pair_features(df_to_pick, 'emb_reconstruction', n_pairs=10000, seed=42)
else:
    print("sampling 5000 pairs")
    df_pairs_contrastive = compute_pair_features(df_to_pick, 'emb', n_pairs=5000, seed=42)
    df_pairs_recon = compute_pair_features(df_to_pick, 'emb_reconstruction', n_pairs=5000, seed=42)

In [ ]:
df_pairs_contrastive['cos_sim_z_score'] = (df_pairs_contrastive['cos_sim'] - df_pairs_contrastive['cos_sim'].mean()) / df_pairs_contrastive['cos_sim'].std()
df_pairs_recon['cos_sim_z_score'] = (df_pairs_recon['cos_sim'] - df_pairs_recon['cos_sim'].mean()) / df_pairs_recon['cos_sim'].std()
df_pairs_recon['cos_sim_z_score'].head()

In [ ]:
import numpy as np

def cohens_d(x, y):
    x = np.array(x)
    y = np.array(y)

    nx, ny = len(x), len(y)

    sx2 = x.var(ddof=1)
    sy2 = y.var(ddof=1)

    pooled = np.sqrt(((nx - 1)*sx2 + (ny - 1)*sy2) / (nx + ny - 2))

    return (x.mean() - y.mean()) / pooled

In [ ]:
def print_correlations(df_pairs, representation_name, sim_score_col='cos_sim'):
    """
    Computes the Spearman correlation of continuous and Cohen's d of discrete fields

    Arguments:
        df_pairs: which df to calculate on (contrastive vs reconstruction)
        representation_name: which representation it is  (purely visual)
        sim_score_col: which similarity score to use
    Returns:
        N/A (prints results)
    """

    # Correlations with cosine similarity
    print("What features correlate with embedding similarity for " + representation_name + "?\n")

    # Continuous features
    for feat in ['edit_dist', 'len_diff']:
        r_spearman, p = spearmanr(df_pairs[feat], df_pairs[sim_score_col])
        print(f"{feat:20s}: Spearman r = {r_spearman:6.3f}, p = {p:.3e}")

    print()

    # Binary features (compare means)
    for feat in ['same_v_trb', 'same_j_trb', 'same_v_paired', 'same_j_paired', 'same_clonotype', 'same_sample']:
        
        group1 = df_pairs.loc[df_pairs[feat], sim_score_col]
        group0 = df_pairs.loc[~df_pairs[feat], sim_score_col]

        d = cohens_d(
            df_pairs[df_pairs[feat]][sim_score_col],
            df_pairs[~df_pairs[feat]][sim_score_col]
        )
        print(f"{feat:20s}: Cohen's d = " + str(d))

In [ ]:
# print_correlations(df_pairs_contrastive, "contrastive", 'cos_sim_z_score')
print_correlations(df_pairs_contrastive, "contra", 'cos_sim_z_score')

In [ ]:
assert(df_pairs_contrastive.shape == df_pairs_recon.shape)

In [ ]:
df_pairs_recon.shape

In [ ]:
def show_graph(df_pairs, xlabel="Edit distance", ylabel="Cosine similarity", title="Title",
               figsize=(8, 6), mode="scatter", gene_suffix="", palette = 'cividis'):
    """
    Customizeable graph plotter with a variety of settings

    Args:
        df_pairs: which embedding pairs dataset to plot (contrastive or reconstruction)
        mode: Which type of plot to generate, must be either "scatter" or "hex"
    Returns:
        N/A, plots graph directly when invoked
    """
    plt.figure(figsize=figsize)
    if mode == "scatter":
        binned = (
            df_pairs
            .assign(bin=lambda x: pd.qcut(x["edit_dist"], q=6))
            .groupby("bin")["cos_sim"]
            .agg(["mean", "std", "count"])
        )
        plt.scatter(df_pairs["edit_dist"], df_pairs["cos_sim"], alpha=0.3, s=10)

        # Add binned means
        binned.reset_index(inplace=True)
        binned["center"] = binned["bin"].apply(lambda x: x.mid)
        plt.plot(binned["center"], binned["mean"], 'r-o', label="Binned mean", markersize=8)

        plt.legend()
        #plt.savefig("edit_dist_vs_cos_sim.png", dpi=150)
    elif mode=="hex":
        plt.hexbin(
            x=df_pairs["edit_dist"+gene_suffix], 
            y=df_pairs["cos_sim"], 
            gridsize=40, 
            cmap=palette,#"inferno"#, mincnt=1
            #bins='log'
            
        )
        plt.colorbar(label="Count")
        sns.regplot(
            data=df_pairs,#.sample(min(len(df_pairs), 5000)),
            x="edit_dist"+gene_suffix,
            y="cos_sim",
            scatter=False,
            lowess=True,   # ← key change
            color="red"
        )
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    #plt.tight_layout()
    plt.show()

### Visualize embedding similarity

In [ ]:
# define which embedding to plot
whichone = "contra."

# can exclude edit dist = 0 pairs for better visualization
exclude_0 = True

df_to_plot = df_pairs_recon.copy()
if "contra" in whichone:
    df_to_plot = df_pairs_contrastive.copy()
    print("contrastive")
else:
    print("recon")
if exclude_0:
    df_to_plot = df_to_plot[df_to_plot['edit_dist'] > 0]
    print("excluding 0")

In [ ]:
show_graph(df_to_plot, figsize=(6, 5), mode="hex", gene_suffix="", title="Embedding similarity vs edit distance ("+ whichone + ", TRA+TRB)",palette='inferno')

In [ ]:
# optionally control for only cells from different patients or different clonotypes to avoid confounding
same_clone_excl = False
same_sample_excl = False

In [ ]:
# pick a tight CDR3 edit dist. band (adjust as needed)
edit_dist_plot = 6
edit_dist_range = 1
df_sub = df_to_plot[(df_to_plot["edit_dist_trb"].between(edit_dist_plot, edit_dist_plot+edit_dist_range))]

if same_clone_excl:
    df_sub = df_sub[df_sub['same_clonotype'] == False].copy()
if same_sample_excl:
    df_sub = df_sub[df_sub['same_sample'] == False].copy()

plt.figure(figsize=(6,5))
plt.xlim(-4, 4)

sns.kdeplot(data=df_sub, x="cos_sim_z_score", hue="same_v_paired", fill=True, common_norm=False, bw_adjust=1.5)

plt.xlabel("Cosine Similarity Z-score")
plt.title("Similarity of same edit distance pairs (" + str(edit_dist_plot) + "-" + str(edit_dist_plot + 1) + "), split by beta J gene usage (" + whichone + ")")

plt.tight_layout()
plt.show()

### ROC-AUC analysis

In [ ]:
df_unique = df_pairs_contrastive[df_pairs_contrastive["same_clonotype"] == False]
#df_unique = df_pairs_recon[df_pairs_recon["same_clonotype"] == False]

In [ ]:
# test this with clonotype effects removed too
df_unique.groupby("same_v_trb")["cos_sim"].describe()

In [ ]:
df_pairs_contrastive.groupby("same_j_trb")["same_clonotype"].mean()

In [ ]:
from sklearn.metrics import roc_auc_score

print(roc_auc_score(df_unique["same_v_trb"], -df_unique["edit_dist_trb"]))
print(roc_auc_score(df_unique["same_v_trb"], -df_unique["edit_dist"]))
roc_auc_score(df_unique["same_v_trb"], df_unique["cos_sim"])

In [ ]:
def split_pairs(arr):
    split_chains = [None] * len(arr)
    for i in range(len(arr)):
        split_chains[i] = arr[i].split(";")
    return split_chains

print(split_pairs("AGG;GGAGC_".split("_")))
print(len(split_pairs("AGG;GGAGC_".split("_"))))
print(len(split_pairs("AGG;GGAGC_".split("_"))[1]))

## CDR3 Levenshtein baseline evaluation

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances
import Levenshtein
from itertools import permutations

def split_pairs(arr):
    split_chains = [None] * len(arr)
    for i in range(len(arr)):
        split_chains[i] = arr[i].split(";")
    return split_chains

def min_dist(a, b):
    n = max(len(a), len(b))

    a_pad = a.copy()
    b_pad = b.copy()

    while len(a_pad) < n:
        a_pad.append("")
    while len(b_pad) < n:
        b_pad.append("")

    best_score = float("inf")

    for perm in permutations(b_pad):
        total = 0
        for i in range(n):
            total += Levenshtein.distance(a_pad[i], perm[i])
        if total < best_score:
            best_score = total

    return int(best_score)


# this thing still works, its just that now that multi-chains have been dropped in QC instead, its a bit overkill
def edit_distance_matrix(sequences, combined=True, a_weight = 1, b_weight = 1, norm = 1):
    """Compute pairwise edit distances"""
    n = len(sequences)
    dist_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i+1, n):
            if combined:
                seq_i = split_pairs(sequences[i].split("_")) # [[ai0, ai1, ... ], [bi0, bi1, ...]]
                seq_j = split_pairs(sequences[j].split("_")) # [[aj0, aj1, ... ], [bj0, bj1, ...]]
                dist_a = min_dist(seq_i[0], seq_j[0]) # compare [ai0, ai1, ...] to [aj0, aj1, ...]
                dist_b = min_dist(seq_i[1], seq_j[1]) # compare [bi0, bi1, ...] to [bj0, bj1, ...]
                dist = (a_weight * dist_a + b_weight * dist_b) / norm # normalize
            else:
                [seq_i] = split_pairs([sequences[i]]) # split_pairs yields [[i0, i1, ...]]
                [seq_j] = split_pairs([sequences[j]]) # split_pairs yields [[j0, j1, ...]]
                dist = min_dist(seq_i, seq_j)

            dist_matrix[i, j] = dist
            dist_matrix[j, i] = dist
    
    return dist_matrix

In [ ]:
# testing our function

seqs = [
    "AAAB_BB",
    "AAC_BCA"
]

print(edit_distance_matrix(seqs, combined=True))
print(Levenshtein.distance('AAAB', 'AAC') + Levenshtein.distance('BB', 'BCA'))

In [ ]:
# On df_eval
sequences = df_to_pick['cdr3_comb'].values
edit_dist_matrix = edit_distance_matrix(sequences, combined=True)

# Convert distance to similarity for consistency
# (smaller distance = higher similarity)
max_dist = edit_dist_matrix.max()
edit_similarity_matrix = 1 - (edit_dist_matrix / max_dist)

# Compute k-NN purity using distances
def knn_purity_from_distance_matrix(dist_matrix, labels, k=10):
    """k-NN purity using precomputed distance matrix"""
    n = len(labels)
    purities = []
    
    for i in range(n):
        # Get k nearest neighbors (excluding self)
        neighbors = np.argsort(dist_matrix[i])[1:k+1]
        same_label = (labels[neighbors] == labels[i]).sum()
        purities.append(same_label / k)
    
    return np.mean(purities)

purity_edit = knn_purity_from_distance_matrix(
    edit_dist_matrix,
    df_to_pick['comb_raw_clonotype_id_t'].values,
    k=10
)

print(f"Edit distance k-NN purity: {purity_edit:.3f}")